In [1]:
import pandas as pd
import numpy as np
import boto3
import io
import json
import joblib
import datetime
import warnings
warnings.filterwarnings('ignore')

import xgboost as xgb
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.metrics import precision_score, recall_score
from scipy import stats

BUCKET_NAME = "churn-mlops-project-cherry"
s3 = boto3.client('s3')

def load_from_s3(bucket, key):
    obj = s3.get_object(Bucket=bucket, Key=key)
    return pd.read_csv(io.BytesIO(obj['Body'].read()))

print("✅ Libraries loaded!")

✅ Libraries loaded!


In [2]:
# This function checks if drift has occurred
# It reads the latest monitoring report from S3
# and decides if retraining is needed

def check_if_retraining_needed():
    try:
        obj = s3.get_object(
            Bucket=BUCKET_NAME,
            Key='monitoring/latest_summary.json'
        )
        summary = json.loads(obj['Body'].read())

        print("📊 Latest Monitoring Summary:")
        print(f"   Checked at:    {summary['checked_at']}")
        print(f"   Max PSI:       {summary['max_psi']}")
        print(f"   Action needed: {summary['action_needed']}")
        print(f"   Recommendation: {summary['recommendation']}")

        return summary['action_needed']

    except Exception as e:
        print(f"No monitoring report found: {e}")
        print("Running retraining as safety measure...")
        return True

# Check if we need to retrain
needs_retraining = check_if_retraining_needed()
print(f"\n🔍 Retraining needed: {needs_retraining}")

📊 Latest Monitoring Summary:
   Checked at:    2026-04-14 06:46:29.768992
   Max PSI:       0.0632
   Action needed: False
   Recommendation: Model is stable

🔍 Retraining needed: False


In [1]:
# This is the full automated retraining pipeline
# It runs automatically when drift is detected
# Does everything: load data, train, evaluate, save

def run_retraining_pipeline():
    print("🚀 Starting automated retraining pipeline...")
    print("="*55)

    # ── Step 1: Load latest data ───────────────────────────
    print("\n📂 Step 1: Loading latest data from S3...")
    X_train = load_from_s3(BUCKET_NAME, 'data/features/X_train.csv')
    X_test  = load_from_s3(BUCKET_NAME, 'data/features/X_test.csv')
    y_train = load_from_s3(BUCKET_NAME, 'data/features/y_train.csv').squeeze()
    y_test  = load_from_s3(BUCKET_NAME, 'data/features/y_test.csv').squeeze()
    print(f"   ✅ Data loaded: {X_train.shape[0]} training rows")

    # ── Step 2: Load old model metrics ────────────────────
    print("\n📊 Step 2: Loading old model metrics...")
    try:
        obj = s3.get_object(
            Bucket=BUCKET_NAME,
            Key='models/latest/metadata.json'
        )
        old_metadata = json.loads(obj['Body'].read())
        old_auc      = old_metadata.get('auc', 0)
        print(f"   Old model AUC: {old_auc}")
    except:
        old_auc = 0
        print("   No old model found. Training fresh.")

    # ── Step 3: Train new model with Optuna ───────────────
    print("\n🔍 Step 3: Running Optuna hyperparameter search...")

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 400),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
            'use_label_encoder': False,
            'eval_metric':      'auc',
            'random_state':     42,
            'n_jobs':           -1
        }
        model = xgb.XGBClassifier(**params)
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)
        y_pred_proba = model.predict_proba(X_test)[:, 1]
        return roc_auc_score(y_test, y_pred_proba)

    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42)
    )
    study.optimize(objective, n_trials=30)
    print(f"   ✅ Best AUC found: {study.best_value:.4f}")

    # ── Step 4: Train final model ──────────────────────────
    print("\n🏋️  Step 4: Training final model...")
    best_params = study.best_params
    best_params['use_label_encoder'] = False
    best_params['eval_metric']       = 'auc'
    best_params['random_state']      = 42
    best_params['n_jobs']            = -1

    new_model = xgb.XGBClassifier(**best_params)
    new_model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)

    y_pred       = new_model.predict(X_test)
    y_pred_proba = new_model.predict_proba(X_test)[:, 1]

    new_auc       = roc_auc_score(y_test, y_pred_proba)
    new_accuracy  = accuracy_score(y_test, y_pred)
    new_f1        = f1_score(y_test, y_pred)
    new_precision = precision_score(y_test, y_pred)
    new_recall    = recall_score(y_test, y_pred)

    print(f"   ✅ New model AUC:      {new_auc:.4f}")
    print(f"   ✅ New model Accuracy: {new_accuracy:.4f}")

    # ── Step 5: Compare old vs new model ──────────────────
    print(f"\n⚖️  Step 5: Comparing old vs new model...")
    print(f"   Old AUC: {old_auc:.4f}")
    print(f"   New AUC: {new_auc:.4f}")

    if new_auc >= old_auc:
        print("   ✅ New model is better! Promoting to production...")
        should_promote = True
    else:
        print("   ⚠️  New model is worse. Keeping old model.")
        should_promote = False

    # ── Step 6: Save new model if better ──────────────────
    if should_promote:
        print("\n💾 Step 6: Saving new model to S3...")
        version = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

        local_path = '/tmp/xgboost_retrained_model.pkl'
        joblib.dump(new_model, local_path)

        model_key = f'models/v_{version}/xgboost_churn_model.pkl'
        s3.upload_file(local_path, BUCKET_NAME, model_key)
        s3.upload_file(local_path, BUCKET_NAME,
                       'models/latest/xgboost_churn_model.pkl')

        # Save new metadata
        new_metadata = {
            'version':    version,
            'auc':        round(new_auc, 4),
            'accuracy':   round(new_accuracy, 4),
            'f1_score':   round(new_f1, 4),
            'precision':  round(new_precision, 4),
            'recall':     round(new_recall, 4),
            'features':   X_train.columns.tolist(),
            'trained_at': str(datetime.datetime.now()),
            'model_key':  model_key,
            'retrained':  True,
            'old_auc':    old_auc
        }
        s3.put_object(
            Bucket=BUCKET_NAME,
            Key='models/latest/metadata.json',
            Body=json.dumps(new_metadata, indent=2)
        )
        print(f"   ✅ New model saved!")
        print(f"   📁 Version: {version}")
        return new_metadata

    else:
        print("\n⏭️  Keeping existing model in production")
        return old_metadata

print("✅ Retraining pipeline function defined!")

✅ Retraining pipeline function defined!


In [4]:
# Run the full automated retraining pipeline!
# This is triggered automatically when drift is detected

if needs_retraining:
    print("🚨 Drift detected — running retraining pipeline!\n")
    result = run_retraining_pipeline()
else:
    print("✅ No retraining needed — model is stable!")
    result = None

print("\n" + "="*55)
print("🎉 Automated Retraining Pipeline Complete!")
if result:
    print(f"\n📊 Final Model Metrics:")
    print(f"   AUC:      {result.get('auc')}")
    print(f"   Accuracy: {result.get('accuracy')}")
    print(f"   F1 Score: {result.get('f1_score')}")
    print(f"   Version:  {result.get('version')}")

✅ No retraining needed — model is stable!

🎉 Automated Retraining Pipeline Complete!


In [5]:
# Save a log of every retraining event
# So you can track model history over time

retraining_log = {
    "timestamp":        str(datetime.datetime.now()),
    "triggered_by":     "drift_detection",
    "retraining_ran":   bool(needs_retraining),
    "new_model_deployed": bool(result and result.get('retrained', False)),
    "final_auc":        result.get('auc') if result else None,
    "final_accuracy":   result.get('accuracy') if result else None,
    "version":          result.get('version') if result else None
}

s3.put_object(
    Bucket=BUCKET_NAME,
    Key=f"logs/retraining/{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.json",
    Body=json.dumps(retraining_log, indent=2)
)

print("✅ Retraining log saved to S3!")
print(f"\n📊 Log entry:")
print(json.dumps(retraining_log, indent=2))
print(f"\n🎉 Step 7 Complete!")
print(f"Your pipeline now automatically retrains when drift is detected!")

✅ Retraining log saved to S3!

📊 Log entry:
{
  "timestamp": "2026-04-14 06:49:18.389274",
  "triggered_by": "drift_detection",
  "retraining_ran": false,
  "new_model_deployed": false,
  "final_auc": null,
  "final_accuracy": null,
  "version": null
}

🎉 Step 7 Complete!
Your pipeline now automatically retrains when drift is detected!
